# House Price — Comparaison des Modèles

Ce notebook teste et compare 6 modèles de régression sur le dataset *Ames Housing*.

**Objectif** : identifier le meilleur modèle baseline avant optimisation.

**Modèles testés** :
- Baseline : DummyRegressor
- Linéaire : LinearRegression
- Non-linéaires : DecisionTree, SVR
- Ensemblistes : RandomForest, XGBoost

## 1. Librairies & Configuration

In [ ]:
# reload modules before executing user code.
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import dill
import matplotlib.pyplot as plt
import missingno as msno
import mlflow
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import pendulum
import ppscore as pps
import seaborn as sns
from loguru import logger
# from pycaret.regression import *
from sklearn import set_config
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (r2_score,
                             mean_squared_error,
                             mean_absolute_percentage_error,
                             max_error,
                            )
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from ydata_profiling import ProfileReport
from yellowbrick.regressor import ResidualsPlot


sys.path.append(str(Path.cwd().parent))
from settings.params import (DATA_DIR_INPUT,
                              DATA_DIR_OUTPUT,
                              MODEL_PARAMS,
                              REPORT_DIR,
                              TIMEZONE,
                              MODEL_NAME,
                              SEED
                             )
from src.make_dataset import load_data
from src.trainer import Trainer

set_config(display="diagram", print_changed_only=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# time in UTC
log_fmt = ("<green>{time:YYYY-MM-DD HH:mm:ss.SSS!UTC}</green> | <level>{level: <8}</level> | "
           "<cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - {message}"
          )
log_config = {
    "handlers": [
        {"sink": sys.stderr, "format": log_fmt},
    ],
}
logger.configure(**log_config)


In [ ]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = Path(PROJECT_DIR, 'data')
DATA_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR = Path(PROJECT_DIR, 'models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
DATA_DIR_INPUT = Path(DATA_DIR, 'input')
DATA_DIR_INPUT.mkdir(exist_ok=True, parents=True)
DATA_DIR_OUTPUT = Path(DATA_DIR, 'output')
DATA_DIR_OUTPUT.mkdir(exist_ok=True, parents=True)

logger.info(f"\nProject directory: {PROJECT_DIR} \nData dir: {DATA_DIR} \nModel dir: {MODEL_DIR}")

In [ ]:
TARGET_NAME = MODEL_PARAMS["TARGET"]
logger.info(f"Target name: {TARGET_NAME}")

## 2. Chargement des données

In [ ]:
data = load_data(dataset_name="house_prices", columns_to_lower=True)

In [ ]:
data = data.astype({
    'overallqual':str, 
    'overallcond':str,
    'garageyrblt':str,
    'yearbuilt':str,
    'yearremodadd':str,
    'mssubclass':str,
    'mosold':str,
    'yrsold':str
})

## 3. Sélection des features — Predictive Power Score (PPS)

Le PPS (Predictive Power Score) mesure la capacité prédictive univariée de chaque variable sur la cible. Contrairement à la corrélation, il capte les relations non-linéaires et asymétriques.

Seuil retenu : **PPS ≥ 0.10** (paramètre `MIN_PPS`).

In [ ]:
# Predictive Power Score (PPS) : https://github.com/8080labs/ppscore/
pps_predictors = pps.predictors(df=data.drop(["id", "yrsold", "yearbuilt", "yearremodadd", "garageyrblt"], axis=1),
                                y=TARGET_NAME, output="df", random_seed=SEED)

In [ ]:
pps_predictors

In [ ]:
# get feature names
logger.info(f"PPS threshold: {MODEL_PARAMS['MIN_PPS']}")
FEATURE_NAMES = pps_predictors.loc[pps_predictors.ppscore.round(2) >= MODEL_PARAMS["MIN_PPS"], "x"].values
if not FEATURE_NAMES.any():
    logger.warning(f"No feature with PPS >= {MODEL_PARAMS['MIN_PPS']}. Consider lowering the threshold.")
    FEATURE_NAMES = MODEL_PARAMS["FEATURES"]
set(FEATURE_NAMES), len(FEATURE_NAMES)

## 4. Configuration MLFlow

Chaque run est enregistré dans l'expérience MLFlow pour traçabilité et comparaison reproductible.

In [ ]:
mlflow.set_tracking_uri("mlruns")
EXPERIMENT_NAME = "house_price_regression_no_log"
mlflow.set_experiment(EXPERIMENT_NAME)
logger.info(f"MLFlow experiment: {EXPERIMENT_NAME}")

## 5. Séparation Train / Test

Division des données : **75% entraînement / 25% test** avec `random_state=43` pour la reproductibilité.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data.loc[:, FEATURE_NAMES],
    data[TARGET_NAME],
    test_size=MODEL_PARAMS["TEST_SIZE"],
    random_state=SEED,
)
logger.info(
    f"\nX train: {x_train.shape}\nY train: {y_train.shape}\n"
    f"X test:  {x_test.shape}\nY test:  {y_test.shape}"
)

## 6. Pipeline de prétraitement

Pipeline sklearn avec traitement séparé pour les variables numériques et catégorielles :
- **Numériques** : imputation médiane → `RobustScaler` (résistant aux outliers)
- **Catégorielles** : imputation constante `"undefined"` → `OneHotEncoder`

In [ ]:
numerical_transformer = [
    SimpleImputer(strategy="median"),
    RobustScaler(),
]
categorical_transformer = [
    SimpleImputer(strategy="constant", fill_value="undefined"),
    OneHotEncoder(drop="if_binary", handle_unknown="ignore"),
]

results = {}  # collecte des métriques test pour la comparaison

## 7. Entraînement des modèles

### 7.1 Modèles Baseline

**DummyRegressor** : prédit toujours la moyenne — sert de borne basse de performance.

In [ ]:
with mlflow.start_run(run_name="DummyRegressor"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=DummyRegressor(strategy="mean"),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "DummyRegressor")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["DummyRegressor"] = test_m
logger.info(f"results so far: {list(results.keys())}")

**LinearRegression** : modèle linéaire classique — hypothèse de relation linéaire entre features et cible.

In [ ]:
with mlflow.start_run(run_name="LinearRegression"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=LinearRegression(),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "LinearRegression")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["LinearRegression"] = test_m
logger.info(f"results so far: {list(results.keys())}")

### 7.2 Modèles Non-Linéaires

**DecisionTree** : partitionnement récursif de l'espace des features. Tend à sur-apprendre sans élagage.

In [ ]:
with mlflow.start_run(run_name="DecisionTree"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=DecisionTreeRegressor(random_state=SEED),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "DecisionTree")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["DecisionTree"] = test_m
logger.info(f"results so far: {list(results.keys())}")

**SVR** (Support Vector Regression) : trouve l'hyperplan qui minimise l'erreur dans un tube ε autour des prédictions.

In [ ]:
with mlflow.start_run(run_name="SVR"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=SVR(),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "SVR")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["SVR"] = test_m
logger.info(f"results so far: {list(results.keys())}")

### 7.3 Modèles Ensemblistes

**Random Forest** : agrège les prédictions de 100 arbres (bagging). Réduit la variance par rapport à un arbre seul.

In [ ]:
with mlflow.start_run(run_name="RandomForest"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=RandomForestRegressor(n_estimators=100, random_state=SEED),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "RandomForest")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["RandomForest"] = test_m
logger.info(f"results so far: {list(results.keys())}")

**XGBoost** : boosting par gradient — construit les arbres séquentiellement en minimisant les résidus du modèle précédent.

In [ ]:
with mlflow.start_run(run_name="XGBoost"):
    trainer = Trainer(
        data=data.loc[:, list(FEATURE_NAMES) + [TARGET_NAME]],
        numerical_transformer=numerical_transformer,
        categorical_transformer=categorical_transformer,
        target=TARGET_NAME,
        test_size=MODEL_PARAMS["TEST_SIZE"],
        cv=3,
        estimator=XGBRegressor(n_estimators=100, random_state=SEED, verbosity=0),
    )
    sk_model, train_m, test_m = trainer.train()
    mlflow.log_param("estimator", "XGBoost")
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    results["XGBoost"] = test_m
logger.info(f"results so far: {list(results.keys())}")

## 8. Comparaison des performances

Classement des modèles par RMSE croissant sur le jeu de test.

In [ ]:
comparison_df = (
    pd.DataFrame(results)
    .T
    .rename_axis("Modèle")
    .sort_values("rmse")
)
display(
    comparison_df
    .style
    .highlight_min(color="lightgreen")
    .highlight_max(color="lightsalmon")
)

best_model_name = comparison_df.index[0]
logger.info(f"Meilleur modèle (RMSE) : {best_model_name}")

In [ ]:
model_types = {
    "DummyRegressor":  "#aec6cf",
    "LinearRegression": "#aec6cf",
    "DecisionTree":     "#ffb347",
    "SVR":              "#ffb347",
    "RandomForest":     "#77dd77",
    "XGBoost":          "#77dd77",
}

models = comparison_df.index.tolist()
bar_colors = [model_types.get(m, "#cccccc") for m in models]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(models, comparison_df["rmse"], color=bar_colors)
axes[0].set_xlabel("RMSE ($)")
axes[0].set_title("RMSE par modèle (↓ meilleur)")
axes[0].invert_yaxis()

axes[1].barh(models, comparison_df["r2"], color=bar_colors)
axes[1].set_xlabel("R²")
axes[1].set_title("R² par modèle (↑ meilleur)")
axes[1].invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#aec6cf", label="Baseline"),
    Patch(facecolor="#ffb347", label="Non-linéaire"),
    Patch(facecolor="#77dd77", label="Ensembliste"),
]
axes[0].legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.show()

---
**Conclusions** :
- La **LinearRegression** obtient le meilleur RMSE sur le test set — les features sélectionnées capturent bien la structure linéaire du prix.
- XGBoost montre un léger sur-apprentissage sans optimisation des hyperparamètres.

➡️ Optimisation dans `house_price_03_optimization.ipynb`.